In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [ ]:
plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
datapath = Path("../../data/solar_panel/")

In [ ]:
train_df = pd.read_csv(datapath / "train.csv")

In [ ]:
test_df = pd.read_csv(datapath / "test.csv")

In [ ]:
train_df.info()

In [ ]:
sns.lineplot(train_df.groupby("가구 수")["총 전력 소비량"].mean())

In [ ]:
train_df.groupby("가구 수")["총 전력 소비량"].mean()

In [ ]:
sns.scatterplot(train_df, x="전체 전기 요금", y="총 전력 소비량")

In [ ]:
train_df.head()

In [ ]:
num_features = ['가스 난방 연료 사용 비율', '전기 난방 연료 사용 비율', '주택 단위 중간 가치', '주거용 인센티브 수',
       '아시아인 비율', '평균 가구원 수', '전체 전기 요금', '위도', '총 전력 소비량',
       '연료유/등유 난방 연료 사용 비율', '비주거용 주정부 인센티브', '일일 태양 복사량',
       '지역 코드']
cat_features = ['가구 수', '타일 수', '65-74세 인구 비율']
len(num_features), len(cat_features)

In [ ]:
train_df.columns

In [ ]:
train_df.isna().sum()

In [ ]:
train_df.median(numeric_only=True)

In [ ]:
train_df = train_df.fillna(value=train_df.median(numeric_only=True))
train_df.info()

In [ ]:
plt.figure(figsize=(5,2))
sns.boxplot(train_df, x="65-74세 인구 비율", y="주택 단위 중간 가치", hue="가구 수")

In [ ]:
test_df = test_df.fillna(value=train_df.median(numeric_only=True))

In [ ]:
train_df["가구 수"].value_counts()

In [ ]:
train_df[cat_features] = train_df[cat_features].ffill()

In [ ]:
train_df.info()

In [ ]:
(train_df["태양광 설치 수"] > 1000).sum()

In [ ]:
test_df[cat_features] = test_df[cat_features].bfill()
test_df[cat_features] = test_df[cat_features].ffill()
test_df.info()

In [ ]:
plt.figure(figsize=(5,2))
sns.histplot(train_df["태양광 설치 수"])
plt.xlim(0, 500)

In [ ]:
# sns.pairplot(train)
# plt.savefig(datapath / "pairplot.png", dpi=600)

In [ ]:
sc = StandardScaler()
sc.fit(train_df[num_features])
X = pd.DataFrame(sc.transform(train_df[num_features]), columns=num_features)
y = train_df["태양광 설치 수"]
test_x = pd.DataFrame(sc.transform(test_df[num_features]), columns=num_features)

In [ ]:
X.shape, y.shape

In [ ]:
onehot = OneHotEncoder(drop="first", sparse_output=False)
onehot.fit(train_df[cat_features])
onehot.get_feature_names_out()

In [ ]:
X[onehot.get_feature_names_out()] = onehot.transform(train_df[cat_features])

In [ ]:
test_x[onehot.get_feature_names_out()] = onehot.transform(test_df[cat_features])

In [ ]:
train_x, valid_x, train_y, valid_y = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=42)

In [ ]:
train_x.shape

In [ ]:
param_grid = {"max_depth": [20,30,40], "min_samples_leaf": [5,10,15]}

In [ ]:
grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
grid.fit(train_x, train_y)

In [ ]:
grid.best_params_

In [ ]:
root_mean_squared_error(valid_y, grid.predict(valid_x))

In [ ]:
root_mean_squared_error(train_y, grid.predict(train_x))

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(valid_y, grid.predict(valid_x))

In [ ]:
r2_score(train_y, grid.predict(train_x))

In [ ]:
submission = pd.read_csv(datapath / 'sample_submission.csv')
submission.head()

In [ ]:
submission['태양광 설치 수'] = grid.predict(test_x)

In [ ]:
submission['태양광 설치 수'] = submission['태양광 설치 수'].astype(int)

In [ ]:
submission.to_csv(datapath / 'submission.csv', index=False, encoding="utf-8-sig")